# EXP-019: Lựa chọn đặc trưng cho Random Forest

- Ngày: 22/11/2026 (tổng hợp notebook).
- Ngày chạy: 22/11/2026.
- Mã thực nghiệm: exp019.
- So với lần trước: exp014 trong mạch trình bày. đối chứng trực tiếp là full68 kiểu exp005 (thay tập đặc trưng).

**Thực nghiệm tương ứng:** [experiment.ipynb](../../experiments/04_additional/EXP-019_RF/experiment.ipynb).

**Mạch thực nghiệm:** [EXP-005](Mai_EXP-005-RF.ipynb) → [EXP-009](Mai_EXP-009_RF_tuning.ipynb) → [EXP-014](Mai_EXP-014_RF_balancing.ipynb) → [EXP-019](Mai_EXP-019_RF_.ipynb).


## 1. Mục tiêu và giả thuyết

**Mục tiêu:** Kiểm tra xem giữ ít đặc trưng quan trọng hơn có giữ nguyên hoặc tăng điểm so với dùng đủ 68 biến không.

**Giả thuyết:** Kỳ vọng bỏ bớt biến ít hữu ích có thể giảm nhiễu. Macro-F1 có thể giữ nguyên hoặc tăng. Kết quả có thể khác giữa hai bài toán.


## 2. Cấu hình

Dùng cấu hình trong `experiment.ipynb` và cấu hình đã lưu khi chạy.

Dùng RF 100 cây cho cả hai phiên bản. Đặt `max_depth=None`, `max_features="sqrt"` và seed 42. Không cân bằng lớp.

Với `full68`, dùng đủ 68 đặc trưng làm đối chứng.

Với `selected`, chọn `k ∈ {16, 32, 48, 68}` bằng 3 inner fold theo người. Dùng `SelectFromModel` với `threshold=-inf` và `max_features=k`. RF học cách xếp hạng biến từ train.

**Thay đổi:** đổi tập đặc trưng đầu vào. Trong EXP-019, dùng RF mặc định full68 kiểu EXP-005 làm đối chứng. Không dùng tiếp mô hình cân bằng của EXP-014.


## 3. Cách thực hiện

Dùng dữ liệu physiological68 gồm 6.189 cửa sổ, 68 đặc trưng và 15 người. Bài toán nhị phân phân biệt không stress/stress. Bài toán ba lớp gồm baseline/stress/amusement.

Chỉ dùng outer train để chọn k bằng 3 inner fold. Học cách điền thiếu, xếp hạng và chọn biến trong train của từng inner fold. Sau đó refit trên 14 người. Kiểm tra trên 1 người. Chạy 30 outer fold. Mỗi fold lưu full68 và selected. Có tổng 60 dòng kết quả.

Chỉ dùng train để học cách xử lý dữ liệu. Tính Accuracy, Macro-F1 và Balanced Accuracy cho từng người test. Sau đó lấy trung bình đều qua 15 người.


## 4. Kết quả

Accuracy là tỷ lệ dự đoán đúng. Macro-F1 là trung bình F1 các lớp. Balanced Accuracy là trung bình recall các lớp. 

| Bài toán | Phiên bản | Accuracy | F1 (macro) | Balanced Accuracy |
|---|---|---|---|---|
| binary | full68 | 0.8900 | 0.8735 | 0.8865 |
| binary | selected | 0.8916 | 0.8757 | 0.8902 |
| multiclass | full68 | 0.7717 | 0.6480 | 0.6700 |
| multiclass | selected | 0.7323 | 0.6024 | 0.6313 |


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

here = Path.cwd()
repo = next(p for p in (here, *here.parents) if (p / 'src/wesad/evaluation.py').is_file())
source = repo / 'outputs/exp019_rf_mai_20261122'
file = source / ('results.csv' if (source / 'results.csv').exists() else 'metrics_by_subject.csv')
scores = pd.read_csv(file)
if 'model' in scores:
    scores = scores.loc[scores.model.eq('rf')]
if 'variant' not in scores:
    scores['variant'] = 'rf'
keys = ['task', 'variant']
display(scores.groupby(keys)[['macro_f1', 'balanced_accuracy']].agg(['mean', 'std']).round(4))


macro_f1         balanced_accuracy        
                        mean     std              mean     std
task       variant                                            
binary     full68     0.8735  0.1440            0.8865  0.1143
           selected   0.8757  0.1440            0.8902  0.1147
multiclass full68     0.6480  0.1297            0.6700  0.1204
           selected   0.6024  0.1478            0.6313  0.1288

### Đủ 68 biến và chọn đặc trưng

![Đủ 68 biến và chọn đặc trưng](../../experiments/04_additional/EXP-019_RF/figures/rf_full68_vs_selected.png)


### Train và test

![Train và test](../../experiments/04_additional/EXP-019_RF/figures/train_vs_test.png)


### Tần suất lựa chọn đặc trưng

![Tần suất lựa chọn đặc trưng](../../experiments/04_additional/EXP-019_RF/figures/rf_feature_selection_frequency.png)


## 5. So sánh với các lần trước

Dùng điểm test trong bảng. Chỉ so sánh các kết quả trong cùng bài toán.

| Bài toán | Lần chạy | Thay đổi | Accuracy | F1 (macro) | Balanced Accuracy |
|---|---|---|---|---|---|
| binary | Đối chứng full68 (kiểu exp005) | Đủ 68 biến | 0.8900 | 0.8735 | 0.8865 |
| binary | exp019, chọn đặc trưng | Chọn k bằng inner CV | 0.8916 | 0.8757 | 0.8902 |
| multiclass | Đối chứng full68 (kiểu exp005) | Đủ 68 biến | 0.7717 | 0.6480 | 0.6700 |
| multiclass | exp019, chọn đặc trưng | Chọn k bằng inner CV | 0.7323 | 0.6024 | 0.6313 |


## 6. Nhận xét

- **Giả thuyết:** thấy chọn biến chỉ giúp tăng nhẹ điểm nhị phân. F1 tăng khoảng 0,0022 và BA tăng khoảng 0,0037. Ba lớp giảm F1 khoảng 0,0456 và BA khoảng 0,0387 so với full68.
- **Quan sát bất thường:** giữ trung bình khoảng 47,5 biến ở nhị phân và 53,1 biến ở ba lớp. Tính số trung bình này qua các fold. Không dùng nó làm số biến cố định để triển khai. Điểm train vẫn gần 1.
- **Lỗi / khó khăn:** các biến tương quan có thể chia sẻ độ quan trọng. Chỉ dùng bảng xếp hạng tổng hợp để mô tả kết quả. Không dùng bảng này để chọn lại biến theo người test. So sánh EXP-019 với full68, không phải mô hình cân bằng EXP-014.


## 7. Bước tiếp theo

1. Tổng hợp bốn hướng RF theo từng bài toán. Không mặc định lần sau tốt hơn lần trước.
2. Cân nhắc điểm test, mức thay đổi điểm, số biến và chi phí tính toán.
3. Nếu kết hợp các thay đổi, làm một thực nghiệm riêng. Chỉ dùng dữ liệu train để chọn tham số và đặc trưng.

**Liên kết:** [EXP-005](Mai_EXP-005-RF.ipynb) → [EXP-009](Mai_EXP-009_RF_tuning.ipynb) → [EXP-014](Mai_EXP-014_RF_balancing.ipynb) → [EXP-019](Mai_EXP-019_RF_.ipynb).
